# Crop Recommendation: Cleaned & Audited Notebook

Predicts the most suitable crop (22 classes) from soil nutrients (N, P, K), temperature, humidity, pH and rainfall.

**What was fixed compared with the original notebook**

| # | Problem in original | Fix here |
|---|---|---|
| 1 | Saved `model.pkl` was not the model that was demoed; two scalers had to be applied in a hidden order | One `Pipeline` (scaler + model) is trained, evaluated, demoed **and** saved: same object everywhere |
| 2 | Label mapping (`crop_dict`) was never saved, so predictions (1-22) could not be decoded | `LabelEncoder` (0-based) is saved next to the pipeline, plus a `metadata.json` |
| 3 | Misleading "data leakage fix" cell and `crop_num` guards for a column that never existed; duplicated split/scaling code | Removed. A single clean preprocessing path |
| 4 | AdaBoost scored 14.5% (default depth-1 stumps cannot handle 22 classes) | AdaBoost now uses depth-3 trees |
| 5 | MinMaxScaler followed by StandardScaler (redundant) | StandardScaler only, inside the pipeline |
| 6 | Models compared on the test set, one 80/20 split, accuracy only | Model selection by 5-fold stratified CV on the training set; test set used once at the end, with a full classification report and confusion matrix |
| 7 | No `random_state`, non-stratified split | Fixed seed everywhere, stratified split |
| 8 | Demo used a training row; scaler feature-name warning | Demo uses a held-out row; input is a DataFrame with proper column names |
| 9 | Junk/duplicate cells, deprecated `distplot` | Removed / replaced with `histplot` |

## 1. Imports & configuration

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

from sklearn.base import clone
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier,
                              GradientBoostingClassifier, RandomForestClassifier)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier

RANDOM_STATE = 42
DATA_PATH = "Crop_recommendation.csv"
ARTIFACT_DIR = Path("model_artifacts")

FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
TARGET = "label"

sns.set_theme(style="whitegrid")

## 2. Load data & basic checks

In [ ]:
df = pd.read_csv(DATA_PATH)

missing_cols = set(FEATURES + [TARGET]) - set(df.columns)
assert not missing_cols, f"Missing columns in CSV: {missing_cols}"

print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Number of crops:", df[TARGET].nunique())
df.head()

In [ ]:
df.info()
df.describe().T

## 3. Exploratory analysis

In [ ]:
counts = df[TARGET].value_counts()
print(f"Samples per crop: min={counts.min()}, max={counts.max()} (balanced: {counts.min() == counts.max()})")

plt.figure(figsize=(8, 6))
sns.heatmap(df[FEATURES].corr(), annot=True, fmt=".2f", cmap="Reds", square=True)
plt.title("Feature correlation")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), FEATURES):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)
for ax in axes.ravel()[len(FEATURES):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

## 4. Features, target encoding & train/test split

The crop names are encoded with a `LabelEncoder` (0-based). The encoder is saved later so predictions can be decoded back to names.
The split is **stratified**, so every crop appears equally in train and test.

In [ ]:
X = df[FEATURES].copy()

le = LabelEncoder()
y = le.fit_transform(df[TARGET])

print("Class mapping:")
for i, name in enumerate(le.classes_):
    print(f"  {i:2d} -> {name}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("\nTrain:", X_train.shape, "| Test:", X_test.shape)

## 5. Compare models with cross-validation

Each model is wrapped in a `Pipeline` with a `StandardScaler`, so scaling is fit **only on the training folds** (no leakage).
Models are compared by 5-fold stratified CV on the **training set only**. The test set is not touched here.

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "GaussianNB": GaussianNB(),
    "KNeighbors": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "ExtraTree": ExtraTreeClassifier(random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "Bagging": BaggingClassifier(random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    # Default AdaBoost (depth-1 stumps) fails on 22 classes (~15% accuracy), so use deeper trees
    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
        n_estimators=200, learning_rate=0.5, random_state=RANDOM_STATE,
    ),
    "SVC": SVC(),
}

def make_pipeline(model):
    return Pipeline([("scaler", StandardScaler()), ("model", clone(model))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, model in models.items():
    scores = cross_val_score(make_pipeline(model), X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    rows.append({"model": name, "cv_mean": scores.mean(), "cv_std": scores.std()})

cv_results = (pd.DataFrame(rows)
                .sort_values("cv_mean", ascending=False)
                .reset_index(drop=True))
cv_results.style.format({"cv_mean": "{:.4f}", "cv_std": "{:.4f}"})

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=cv_results, y="model", x="cv_mean", color="seagreen")
plt.xlim(cv_results["cv_mean"].min() - 0.02, 1.0)
plt.xlabel("5-fold CV accuracy (training set)")
plt.ylabel("")
plt.tight_layout()
plt.show()

best_name = cv_results.loc[0, "model"]
print("Best model by CV:", best_name)
print("Note: differences of a few tenths of a percent between the top models are within noise (see cv_std).")

## 6. Final evaluation on the held-out test set (used once)

In [ ]:
best_pipe = make_pipeline(models[best_name])
best_pipe.fit(X_train, y_train)

y_pred = best_pipe.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"{best_name} test accuracy: {test_acc:.4f}\n")
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=3))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(13, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion matrix: {best_name}")
plt.tight_layout()
plt.show()

errors = X_test[y_test != y_pred].copy()
errors["actual"] = le.inverse_transform(y_test[y_test != y_pred])
errors["predicted"] = le.inverse_transform(y_pred[y_test != y_pred])
print(f"Misclassified samples: {len(errors)} of {len(X_test)}")
errors

In [ ]:
perm = permutation_importance(best_pipe, X_test, y_test, n_repeats=10,
                              random_state=RANDOM_STATE, n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values()

plt.figure(figsize=(7, 4))
importance.plot.barh(color="seagreen")
plt.xlabel("Drop in accuracy when feature is shuffled")
plt.title("Permutation importance (test set)")
plt.tight_layout()
plt.show()

## 7. Recommendation function

`recommend_crop` takes raw (unscaled) values, builds a DataFrame with the correct column names (no feature-name warning) and returns the top predictions.
It warns if an input falls outside the range seen during training.

In [ ]:
feature_ranges = X.agg(["min", "max"])

def recommend_crop(N, P, K, temperature, humidity, ph, rainfall, top_k=3, pipeline=None):
    pipeline = pipeline if pipeline is not None else best_pipe
    sample = pd.DataFrame([[N, P, K, temperature, humidity, ph, rainfall]], columns=FEATURES)

    for col in FEATURES:
        lo, hi = feature_ranges.loc["min", col], feature_ranges.loc["max", col]
        if not (lo <= sample.loc[0, col] <= hi):
            warnings.warn(f"{col}={sample.loc[0, col]} is outside the training range [{lo:.2f}, {hi:.2f}]; "
                          "prediction may be unreliable.")

    result = {"recommended": le.inverse_transform(pipeline.predict(sample))[0], "top_k": None}
    if hasattr(pipeline, "predict_proba"):
        proba = pipeline.predict_proba(sample)[0]
        idx = np.argsort(proba)[::-1][:top_k]
        result["top_k"] = [(le.classes_[i], round(float(proba[i]), 3)) for i in idx]
    return result

# Demo on a HELD-OUT row (the model has not seen it)
row = X_test.iloc[0].to_dict()
print("Input :", row)
print("Actual:", le.inverse_transform([y_test[0]])[0])
print("Output:", recommend_crop(**row))

In [ ]:
# Custom example
recommend_crop(N=90, P=42, K=43, temperature=21.0, humidity=82.0, ph=6.5, rainfall=203.0)

## 8. Train the deployment model on all data & save artifacts

After evaluation, the chosen pipeline is re-fit on **all** 2,200 rows (more data for the deployed model). The test accuracy above remains the honest estimate of its performance.

Saved files (in `model_artifacts/`):
- `crop_pipeline.joblib`: scaler + model in one object (pass a raw DataFrame, get label indices)
- `label_encoder.joblib`: converts indices back to crop names
- `metadata.json`: feature order, classes, valid input ranges, scikit-learn version, scores

In [ ]:
final_pipe = clone(best_pipe).fit(X, y)

ARTIFACT_DIR.mkdir(exist_ok=True)
joblib.dump(final_pipe, ARTIFACT_DIR / "crop_pipeline.joblib")
joblib.dump(le, ARTIFACT_DIR / "label_encoder.joblib")

metadata = {
    "model": best_name,
    "features": FEATURES,
    "classes": list(le.classes_),
    "feature_ranges": {c: [float(feature_ranges.loc["min", c]), float(feature_ranges.loc["max", c])] for c in FEATURES},
    "cv_accuracy": float(cv_results.loc[0, "cv_mean"]),
    "test_accuracy": float(test_acc),
    "sklearn_version": sklearn.__version__,
    "random_state": RANDOM_STATE,
}
with open(ARTIFACT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", sorted(p.name for p in ARTIFACT_DIR.iterdir()))

In [ ]:
# Verify the saved files reload and behave identically
loaded_pipe = joblib.load(ARTIFACT_DIR / "crop_pipeline.joblib")
loaded_le = joblib.load(ARTIFACT_DIR / "label_encoder.joblib")

assert np.array_equal(loaded_pipe.predict(X), final_pipe.predict(X)), "Reloaded model differs!"
assert list(loaded_le.classes_) == list(le.classes_), "Reloaded encoder differs!"

demo = pd.DataFrame([[90, 42, 43, 21.0, 82.0, 6.5, 203.0]], columns=FEATURES)
print("Reload check passed. Example prediction:", loaded_le.inverse_transform(loaded_pipe.predict(demo))[0])

## 9. How to use the saved model elsewhere (e.g. Flask / Streamlit app)

```python
import joblib, pandas as pd

pipe = joblib.load("model_artifacts/crop_pipeline.joblib")
le   = joblib.load("model_artifacts/label_encoder.joblib")

features = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
sample = pd.DataFrame([[90, 42, 43, 21.0, 82.0, 6.5, 203.0]], columns=features)
print(le.inverse_transform(pipe.predict(sample))[0])
```

Load with the same scikit-learn version recorded in `metadata.json`, because joblib/pickle files can break across versions.

## 10. Limitations

- This Kaggle crop dataset is very clean and its classes are almost perfectly separable (the original notebook also hit ~99%), so high accuracy here **does not guarantee** correct recommendations on real farms. Real data has noise, soil type, season, location and market factors that are not in these 7 features.
- Inputs outside the training range (see `metadata.json`) should be treated as unreliable.
- Treat the output as decision support, not a replacement for local agronomic advice.